### =====================================================
### README
### =====================================================
#### Nome: Plotagem de perfis topográficos por AOI
#
#### Descrição:
##### Este script lê shapefiles de perfis topográficos armazenados no Drive compartilhado Batimetrias_Babitonga, dentro da pasta PERFIS.
#
#### Para cada AOI, o script:
1. Procura arquivos com prefixo "profile_shape_"
2. Identifica o ano de referência no nome do arquivo
3. Separa os perfis pelo campo ID
4. Gera um gráfico por perfil, sobrepondo todos os anos disponíveis
#
#### Regras do gráfico:
- textos em português
- título = ID do perfil
- todas as curvas em preto
- diferenciação por tipo de traço
- legenda mostrando apenas o ano
- mesmos limites de eixo para todos os perfis da mesma AOI
- se houver valores nulos, o traçado ficará com gap
#
#### Saída:
 - arquivos PNG salvos no Drive, em subpastas por AOI
# ============================================================


In [5]:
# ============================================
# 1. Instalar bibliotecas
# ============================================
!pip install -q geopandas pyogrio matplotlib shapely

# ============================================
# 2. Importar bibliotecas
# ============================================
import os
import re
import glob
import math
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from google.colab import drive

In [2]:
# ============================================
# 3. Montar Google Drive
# ============================================
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:

# ============================================
# 4. Caminhos principais
# ============================================
ROOT = "/content/drive/Shareddrives/Batimetrias_Babitonga/PERFIS"
OUTPUT_ROOT = os.path.join(ROOT, "_FIGURAS_PERFIS")
os.makedirs(OUTPUT_ROOT, exist_ok=True)

# Se quiser limitar às AOIs conhecidas, deixe assim:
AOI_FOLDERS = [
    "01_AOI01",
    "02_AOI02",
    "03_AOI03"
]

# Se preferir detectar automaticamente, use:
# AOI_FOLDERS = sorted([f for f in os.listdir(ROOT) if os.path.isdir(os.path.join(ROOT, f))])

# ============================================
# 5. Funções auxiliares
# ============================================
def find_profile_shapefiles(aoi_dir):
    return sorted(
        glob.glob(os.path.join(aoi_dir, "**", "profile_shape_*.shp"), recursive=True)
    )

def extract_year_from_layer(layer_value):
    """
    Extrai ano da coluna layer.
    Exemplo:
    ProfileTool_sup_delta_2014_band_1 -> 2014
    """
    if pd.isna(layer_value):
        return None
    m = re.search(r"(19\d{2}|20\d{2})", str(layer_value))
    if m:
        return int(m.group(1))
    return None

def compute_profile_distance_from_points(sub):
    """
    Reconstrói a distância ao longo do perfil usando PCA para ordenar os pontos
    ao longo do eixo principal.
    """
    if sub.empty:
        return None, None

    # coordenadas dos pontos
    coords = np.array([(geom.x, geom.y) for geom in sub.geometry])

    if coords.shape[0] < 2:
        return None, None

    # ordenação ao longo do eixo principal
    pca = PCA(n_components=1)
    coord_1d = pca.fit_transform(coords).flatten()
    order = np.argsort(coord_1d)

    coords_sorted = coords[order]
    values_sorted = pd.to_numeric(sub["Value"], errors="coerce").to_numpy(dtype=float)[order]

    # distância acumulada
    dist = [0.0]
    for i in range(1, len(coords_sorted)):
        d = math.hypot(
            coords_sorted[i, 0] - coords_sorted[i - 1, 0],
            coords_sorted[i, 1] - coords_sorted[i - 1, 1]
        )
        dist.append(dist[-1] + d)

    dist = np.array(dist, dtype=float)

    # mantém NaN para gerar gaps
    return dist, values_sorted

def compute_area_limits(profile_dict):
    xs = []
    ys = []

    for pid, year_dict in profile_dict.items():
        for year, (x, y) in year_dict.items():
            if x is None or y is None:
                continue
            xs.extend(list(np.asarray(x)[~np.isnan(x)]))
            ys.extend(list(np.asarray(y)[~np.isnan(y)]))

    if len(xs) == 0 or len(ys) == 0:
        return None

    xmin, xmax = np.min(xs), np.max(xs)
    ymin, ymax = np.min(ys), np.max(ys)

    xpad = 0.03 * (xmax - xmin) if xmax > xmin else 1
    ypad = 0.05 * (ymax - ymin) if ymax > ymin else 0.5

    return (xmin - xpad, xmax + xpad, ymin - ypad, ymax + ypad)

In [8]:
# ============================================
# 6. Leitura e organização dos dados
# ============================================
all_data = {}

for aoi in AOI_FOLDERS:
    aoi_dir = os.path.join(ROOT, aoi)

    if not os.path.isdir(aoi_dir):
        print(f"[AVISO] AOI não encontrada: {aoi_dir}")
        continue

    shp_files = find_profile_shapefiles(aoi_dir)

    if len(shp_files) == 0:
        print(f"[AVISO] Nenhum shapefile profile_shape_ encontrado em {aoi}")
        continue

    print(f"\n{aoi}: {len(shp_files)} shapefiles encontrados")

    aoi_profiles = {}

    for shp in shp_files:
        try:
            gdf = gpd.read_file(shp)
        except Exception as e:
            print(f"  [AVISO] Erro ao ler {os.path.basename(shp)}: {e}")
            continue

        if gdf.empty:
            print(f"  [AVISO] Shapefile vazio: {os.path.basename(shp)}")
            continue

        # checagem de colunas obrigatórias
        required_cols = ["Value", "ID", "layer"]
        missing = [c for c in required_cols if c not in gdf.columns]
        if missing:
            print(f"  [AVISO] Colunas faltando em {os.path.basename(shp)}: {missing}")
            continue

        # extrai ano da coluna layer
        gdf["year"] = gdf["layer"].apply(extract_year_from_layer)

        if gdf["year"].isna().all():
            print(f"  [AVISO] Nenhum ano identificado em {os.path.basename(shp)}")
            continue

        # processa por perfil e ano
        for pid in sorted(gdf["ID"].dropna().unique()):
            sub_pid = gdf[gdf["ID"] == pid].copy()

            for year in sorted(sub_pid["year"].dropna().unique()):
                sub = sub_pid[sub_pid["year"] == year].copy()

                x, y = compute_profile_distance_from_points(sub)
                if x is None or y is None:
                    print(f"  [AVISO] Não foi possível extrair perfil | arquivo={os.path.basename(shp)} | ID={pid} | ano={year}")
                    continue

                aoi_profiles.setdefault(str(pid), {})
                aoi_profiles[str(pid)][int(year)] = (x, y)

    all_data[aoi] = aoi_profiles

# ============================================
# 7. Plotagem
# ============================================
line_styles = [
    "-", "--", "-.", ":",
    (0, (1, 1)),
    (0, (5, 1)),
    (0, (3, 1, 1, 1)),
    (0, (6, 2)),
    (0, (2, 2)),
    (0, (8, 2, 1, 2))
]

for aoi, profile_dict in all_data.items():
    if len(profile_dict) == 0:
        print(f"[AVISO] Nenhum perfil utilizável em {aoi}")
        continue

    limits = compute_area_limits(profile_dict)
    if limits is None:
        print(f"[AVISO] Não foi possível calcular limites para {aoi}")
        continue

    xmin, xmax, ymin, ymax = limits

    out_dir = os.path.join(OUTPUT_ROOT, aoi)
    os.makedirs(out_dir, exist_ok=True)

    print(f"\nGerando figuras para {aoi}...")

    for pid, year_dict in sorted(profile_dict.items(), key=lambda kv: kv[0]):
        years_sorted = sorted(year_dict.keys())
        style_map = {yr: line_styles[i % len(line_styles)] for i, yr in enumerate(years_sorted)}

        fig, ax = plt.subplots(figsize=(8, 4.5), dpi=200)

        for yr in years_sorted:
            x, y = year_dict[yr]

            ax.plot(
                x,
                y,
                color="black",
                linewidth=1.2,
                linestyle=style_map[yr],
                label=str(yr)
            )

        ax.set_title(str(pid), fontsize=12)
        ax.set_xlabel("Distância ao longo do perfil (m)")
        ax.set_ylabel("Cota topográfica (m)")

        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)

        ax.grid(True, alpha=0.25)
        ax.legend(title=None, frameon=True, framealpha=0.95, fontsize=9)

        plt.tight_layout()

        out_png = os.path.join(out_dir, f"perfil_{pid}.png")
        fig.savefig(out_png, dpi=300, bbox_inches="tight")
        plt.close(fig)

    print(f"  [OK] Figuras salvas em: {out_dir}")

print("\nConcluído.")


01_AOI01: 1 shapefiles encontrados

02_AOI02: 1 shapefiles encontrados

03_AOI03: 1 shapefiles encontrados

Gerando figuras para 01_AOI01...
  [OK] Figuras salvas em: /content/drive/Shareddrives/Batimetrias_Babitonga/PERFIS/_FIGURAS_PERFIS/01_AOI01

Gerando figuras para 02_AOI02...
  [OK] Figuras salvas em: /content/drive/Shareddrives/Batimetrias_Babitonga/PERFIS/_FIGURAS_PERFIS/02_AOI02

Gerando figuras para 03_AOI03...
  [OK] Figuras salvas em: /content/drive/Shareddrives/Batimetrias_Babitonga/PERFIS/_FIGURAS_PERFIS/03_AOI03

Concluído.
